In [1]:
from pathlib import Path
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

In [2]:
def first_cap_per_cell_type(adata, cap=10_000, cell_type_key="cell_type", seed=0):
    rng = np.random.default_rng(seed)

    ct = adata.obs[cell_type_key].to_numpy()
    keep_mask = np.zeros(adata.n_obs, dtype=bool)

    for c in np.unique(ct):
        idx = np.where(ct == c)[0]
        if idx.size <= cap:
            keep_mask[idx] = True
        else:
            keep_idx = rng.choice(idx, size=cap, replace=False)
            keep_mask[keep_idx] = True

    keep_idx = np.sort(np.where(keep_mask)[0])  # preserve original order
    return adata[keep_idx].copy()

## Single nuclei data

In [2]:
BASE = Path("/project/Wellcome_Discovery/datashare/lturiano/GENESIS/data")  # <-- change to your base
pattern = "*_sn_filt_1.h5ad"

In [3]:
files = sorted(BASE.rglob(pattern))
print("Found files:", len(files))

for f in files:
    print(" -", f)

Found files: 6
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/all_organs_sn_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/breast/breast_sn_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/heart/heart_sn_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/kidney/kidney_sn_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/liver/liver_sn_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/lung/lung_sn_filt_1.h5ad


In [ ]:
# read + add organ label
adatas = []
for f in files:
    organ = f.name.replace("_sn_filt_1.h5ad", "")
    a = sc.read_h5ad(f)
    adatas.append(a)
    
# concatenate:
# - join="inner" keeps only common genes across all organs (safest)
# - label adds a batch key (organ); index_unique avoids obs name collisions
adata_all = ad.concat(
    adatas,          # dict or list
    join="inner",
    axis=0,
    index_unique=None,  # <- keeps original obs_names unchanged
    merge="first"
)

/package/python-cbrg/current/3.11.14/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
adata_all

In [ ]:
adata_all.obs["cell_type"].value_counts()

In [ ]:
adata_all = first_cap_per_cell_type(adata_all, cap=20_000, cell_type_key="cell_type", seed=42)

In [7]:
adata_all.write_h5ad(BASE / "all_organs_sn_filt_1.h5ad", compression="gzip")

## Single cell data

In [3]:
BASE = Path("/project/Wellcome_Discovery/datashare/lturiano/GENESIS/data")  # <-- change to your base
pattern = "*_sc_filt_1.h5ad"

In [4]:
files = sorted(BASE.rglob(pattern))
print("Found files:", len(files))

for f in files:
    print(" -", f)

Found files: 4
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/breast/breast_sc_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/heart/heart_sc_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/kidney/kidney_sc_filt_1.h5ad
 - /project/Wellcome_Discovery/datashare/lturiano/GENESIS/data/liver/liver_sc_filt_1.h5ad


In [5]:
# read + add organ label
adatas = []
for f in files:
    organ = f.name.replace("_sc_filt_1.h5ad", "")
    a = sc.read_h5ad(f)
    adatas.append(a)
    
# concatenate:
# - join="inner" keeps only common genes across all organs (safest)
# - label adds a batch key (organ); index_unique avoids obs name collisions
adata_all = ad.concat(
    adatas,          # dict or list
    join="inner",
    axis=0,
    index_unique=None,  # <- keeps original obs_names unchanged
    merge="first"
)

In [6]:
adata_all

AnnData object with n_obs × n_vars = 198202 × 29144
    obs: 'cell_type', 'organ'
    var: 'feature_name'

In [7]:
adata_all.obs["cell_type"].value_counts()

cell_type
endothelial cell       29954
epithelial cell        20000
T cell                 15269
fibroblast             14213
macrophage             13501
lymphocyte             11697
myeloid cell           10406
mural cell             10000
duct cell              10000
muscle cell            10000
hepatocyte             10000
basal cell             10000
B cell                  8108
natural killer cell     7326
monocyte                6695
dendritic cell          4433
endothelial             2615
myocyte                 1924
neural cell             1114
mast cell                690
mesothelial cell         195
erythroid                 61
adipocyte                  1
Name: count, dtype: int64

In [8]:
adata_all = first_cap_per_cell_type(adata_all, cap=20_000, cell_type_key="cell_type", seed=42)

In [9]:
adata_all.write_h5ad(BASE / "all_organs_sc_filt_1.h5ad", compression="gzip")